# 03 · Agreement, adjudication → *your* gold set

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/egumasa/lda2-final-template/blob/main/notebooks/03_annotate.ipynb)

The part no model can do for you, and the part the Q&A will ask about.

```
  01_build_pool_<track>  →  02_sample  →▶ 03_annotate  →  04_prompt  →  05_report
```

| | |
|---|---|
| **Reads** | `data/gold/<track>_<group>_sample.json` and the sheet (both from 02) |
| **Writes** | `data/gold/<track>_<group>_gold.json`, and its `_dev.json` / `_test.json` split |

---

**Come here when both coders have finished.** Notebook 02 drew the sample and made the sheet; this one turns two people's labels into one gold set.

The published labels are somebody else's judgment. You re-annotated the sample blind; now you find out how far apart the two of you were, and argue out the rows you disagreed on.

What comes out is *your* gold set — and the disagreements tell you which label boundaries are genuinely fuzzy. That is what lets you say, later, whether a model's miss is the **model's** fault or the **scheme's**. Nothing else in the project can tell you that, and notebook 05 asks you for it directly.

It ends by drawing one more line: which of your annotated items you are allowed to *look at* while you write prompts, and which are held back for the number you report. One sheet, one adjudication, then a split — it costs no extra coding.

## Setup — run this first

This cell mounts your Google Drive and finds your group's shared folder, `lda2-final-template`. Everything the project produces — the pool, the gold set, your prompts, the outputs — is an ordinary file in there, which is what makes it survive the runtime resetting *and* lets the rest of your group see it.

**One member sets the folder up once:**

1. That member runs the `git clone` line this cell prints if the folder is missing, which puts it in their own Drive.
2. They share it with the group (right-click ▸ *Share*), with edit access.
3. Everyone else opens *Shared with me*, right-clicks the folder, and chooses **Add shortcut to Drive** ▸ *My Drive*.

Keep that shortcut's name exactly `lda2-final-template`. It is what makes the same path work for all of you — if Drive renames it to `lda2-final-template (1)`, this cell will not find it.

From then on, open notebooks from the folder itself (*File ▸ Open notebook ▸ Drive*) rather than from the GitHub badge, so you are working on your group's copy and not a fresh one.

**Looking inside a helper.** The functions this cell imports are defined in `scripts/`. Colab cannot jump to a function's definition the way an editor can, so to read one, call `show` on it — for example `show(save_json)`. It prints the file, the line the function starts on, and the code that runs. You can also open `scripts/pipeline.py` yourself from the **Files** panel on the left.

In [ ]:
# ------------------------------------------------------------------
# SETUP — run me first. You are not expected to read it.
# ------------------------------------------------------------------
# This cell is plumbing, and it is the only cell in the project that is.
# It finds your group's shared folder in Google Drive, because everything
# this project keeps goes in there: a Colab runtime is wiped when it resets,
# and nobody else in your group can see inside it. Then it makes the
# project's own code importable. Run it and move on; nothing below asks you
# to have understood it.

FOLDER = "lda2-final-template"     # the shared folder, in every member's Drive

import os, sys

PROJECT = ".."                              # running locally: it is just above us

try:
    from google.colab import drive           # only exists inside Colab
except ImportError:
    pass
else:
    drive.mount("/content/drive")
    PROJECT = "/content/drive/MyDrive/" + FOLDER
    if not os.path.isdir(PROJECT):
        raise RuntimeError(
            "Could not find " + PROJECT + "\n\n"
            "Setting the folder up for your group? Run this in a new cell:\n"
            "  !git clone https://github.com/egumasa/lda2-final-template.git "
            + PROJECT + "\n"
            "then share the folder with the rest of your group.\n\n"
            "Someone else already did? Open Drive, find the folder under "
            "'Shared with me', right-click it, and choose 'Add shortcut to "
            "Drive'. Keep the name exactly " + FOLDER + ".")
    # Work inside the project folder, where the notebooks live.
    os.makedirs(PROJECT + "/notebooks", exist_ok=True)
    os.chdir(PROJECT + "/notebooks")

# scripts/ and config.py, by their real paths - so they are found from wherever
# this notebook happens to be working.
sys.path.append(PROJECT)
sys.path.append(PROJECT + "/scripts")

# Re-read config.yaml every time this cell runs. Without the reload, Python
# hands back the settings it read the FIRST time, and editing config.yaml
# would appear to do nothing until you restarted the runtime.
import importlib
import config
importlib.reload(config)

# Named one by one rather than with `import *`, so that every name a cell
# below uses can be traced back to the file it came from — config.yaml for
# these, scripts/ for the rest.
from config import (TRACK, GROUP, RUN, SEED, N_PER_CLASS, DEV, CODERS,
                    MEMBERS, LABELS_ORDER, ROOT, OUT_DIR,
                    POOL_PATH, DEMO_POOL_PATH, SAMPLE_PATH, GOLD_PATH,
                    DEV_PATH, TEST_PATH, DISAGREED_PATH, PRED_PATH,
                    ROUNDS_PATH, TESTLOG_PATH,
                    PROMPT_FILE, SHEET_PATH, TRIAGE_PATH, describe)

# The Google Sheets round trip is plumbing, so it is imported. The judgment it
# exists to support is not in any of these files.
# `show` prints the source of any of these: show(save_json)
from pipeline import load_gold, label_set, save_json, split_dev_test, show
from annotate import (remembered_sheet, load_coder_sheets, to_canonical,
                      annotator_agreement, disagreements,
                      compare_to_published)

describe()                  # what this notebook is working on


> **Everything above comes from `config.yaml`** — one small file at the top of the repo, which you edit once as a group, and the only file in the plumbing you touch. That is deliberate: the seed that drew your sample has to be the seed you report, and five copies of a number in five notebooks is five chances for them to disagree. Your settings are also the filenames — `track: cars50`, `group: kimura`, `run: v1` means this notebook reads and writes `cars50_kimura_v1_...`. If the line it just printed is not your track, your group and your seed, fix `config.yaml` and re-run this cell.

## First — your sample, back from the file

Notebook 02 saved it, and this is the moment that was for. Days have passed, the runtime that drew the sample is long gone, and the person running this cell may not be the person who ran 02.

It matters that this is a **load and not a redraw**: the sheet your coders filled in was built from these exact forty items, and adjudication puts their labels back onto them one by one. `to_canonical` also uses this list to restore what the sheet does not carry — on `cars50` and `raamove`, the passage each sentence came from.

In [ ]:
sampled = load_gold(SAMPLE_PATH)
LABELS = label_set(sampled)

print(len(sampled), "items ·", LABELS)

# These still carry the PUBLISHED label. Ignore it for now — you compare
# against it in step 4, once your own labels are settled.


## Then — the annotation sheet, found again

Now we look up the sheet notebook 02 created, from the small file it wrote the link to. That file is why the link is not lost: whoever runs this notebook need not be the person who ran notebook 02, and need not still have that cell's output on screen.

If this prints *none saved yet*, notebook 02's sheet step has not been run — or was run by somebody whose copy of the folder is not this one.

In [ ]:
SHEET_ID = remembered_sheet(SHEET_PATH)

# Working on a sheet someone made before this file existed? Paste its URL (or
# just the long id from it) here instead:
# SHEET_ID = ""

print("sheet:", SHEET_ID or "-- none saved yet: run notebook 02 --")


## Step 1 — Measure agreement

Each coder has their **own tab**, so the first thing to do is line them up side by side. `load_coder_sheets` reads one tab per name you give it and joins them by item id into a single table — one column per coder, plus `Final`.

> **Who annotated?** Change the list if your group is not two people. If a third coder joined, duplicate an **empty** tab in the sheet (right-click ▸ *Duplicate*), rename it `CoderC`, and add `"CoderC"` to the list. You did not have to decide this when the sheet was made, and you do not have to rebuild anything now.

Then the numbers. With **two** coders: raw percent agreement, Cohen's κ (agreement corrected for what you would get by guessing), and a coder-vs-coder confusion matrix whose off-diagonal cells show *which* label pairs you confuse. With **three or more**: Fleiss' κ for the group as a whole, then Cohen's κ for every pair, then the matrix for the pair that agreed least — which is usually where your scheme is leaking.

**Write these down now** — they are report section 1, and they do not survive a runtime reset. A κ around .8 is strong; around .4 means the scheme, not the annotators, is doing something wrong. Either is a reportable finding. A low κ you can explain beats a high one you cannot.

These are the Day 2 S5 D–E calls, with the coder names added; they live in `scripts/annotate.py`. Run the cell once **every** coder's tab is filled in — rows that not everyone labelled are dropped from the comparison.

**If it warns that two coders gave every item the same label**, somebody duplicated a tab that had already been filled in. That agreement is a copy rather than a measurement, and it has to be fixed before you report anything.

In [ ]:
# ══ STEP 1 · Measure agreement ════════════════════════════════════════════
# Reads one tab per coder, lines them up by item id, and prints how often you
# agreed, corrected for chance.
# Creates: rows

# ✏️ this runs as written — the work is deciding whether it should

# CODERS comes from config.yaml, so notebook 05 finds the same tabs. Add a
# third name there if a third person joined.
rows = load_coder_sheets(SHEET_ID, CODERS)       # one read per tab, merged by ID

annotator_agreement(rows, coders=CODERS)         # prints agreement and κ


### Now the rows you have to talk about

`disagreements` hands back a table of the rows your coders labelled differently. That is your adjudication list for step 2.

The last line of the cell is just the name `disagreed`, with no `print`. In a notebook, the value of the last line of a cell is displayed automatically — and for a table that display is much easier to read than `print` would give you. Add a line after it and the table stops appearing, which is the one thing to watch out for.

The table is saved as well as shown. It comes back in notebook 05, where the rows your coders argued about are what you check the model's errors against — and saving it here means 05 does not have to sign back in to the sheet and derive the same table a second time. It also means that step still works after the sheet has been deleted, or its owner has left.

In [ ]:
disagreed = disagreements(rows, coders=CODERS)
save_json(disagreed.to_dict("records"), DISAGREED_PATH,
          what="rows your coders disagreed on")
disagreed

## Step 2 — Adjudicate

Go back to the sheet and fill in `Final` for **every** row:

- Where you agreed, `Final` is that label.
- Where you did not, talk it out and decide. If you cannot agree, the scheme is underspecified — write down *why* in `Note` and pick one. That note is worth more to your report than the label is.

Then re-read the sheet and canonicalise it. `to_canonical` reports blanks and invalid labels rather than silently dropping them; fix them in the sheet and re-run until it says **0 blank, 0 invalid**. A blank row is an item that has gone missing from your study without telling you.

The cell re-reads the sheet first, because `rows` from step 1 was fetched before you filled `Final` in. And it passes `source=sampled`: gold is rebuilt from the **sheet**, which carries only the id, the text and your label, so anything else the item had — on `cars50` and `raamove`, its passage — is put back from `sampled` by id. On the other tracks that argument does nothing.

Both calls are the Day 2 S5 step F ones. `to_canonical` is in `scripts/annotate.py`.

In [ ]:
# ══ STEP 2 · Adjudicate, then canonicalise ════════════════════════════════
# Re-reads the sheet now that Final is filled in, and turns it into your gold
# set — reporting any row that is blank or has a label it does not recognise.
# Creates: gold

# ✏️ this runs as written — the work is deciding whether it should

# Re-read: `rows` from step 1 was fetched before you filled in Final.
rows = load_coder_sheets(SHEET_ID, CODERS)

gold = to_canonical(rows, LABELS, source=sampled)   # re-attaches what the sheet drops


## Step 3 — Where do you differ from the published labels?

Now — and only now, with your own labels settled — look at what the corpus said. `compare_to_published` matches by text and shows you every row where your group landed somewhere else.

**Disagreement here is not an error.** You annotated forty items carefully against a scheme you had thought about; the original annotators worked at scale under different guidelines. Where you differ, one of three things is true, and saying which is exactly the analytical work this project is for:

1. **Your scheme drifted** from theirs — you read a category boundary differently. Say where.
2. **The item is genuinely ambiguous** — it would split any pair of annotators.
3. **One of you is wrong.** It happens, in both directions.

This table is report section 1, and it is the one that most often produces a sentence worth saying out loud in the Q&A. Pick two or three rows and write down which of the three cases above they are — now, while you still remember the argument you had about them.

The comparison runs against `sampled`, not `pool`: sampling renumbered the ids, so `pool` would line your item 7 up against a completely different sentence. This is the Day 2 S5 step F call; it is in `scripts/annotate.py`.

In [ ]:
# ══ STEP 3 · Compare against the published labels ═════════════════════════
# Shows every row where your group's label and the corpus's label differ.
# Creates: differences

# ✏️ this runs as written — the work is deciding whether it should

differences = compare_to_published(gold, sampled)   # sampled, not pool: same 40 items
differences


## Save it — this is the handoff

This file is the single most valuable thing your group makes all week — hours of judgment, and the only thing in the project that could not have been produced by a script. Every number in notebooks 04 and 05 is measured against it, and it goes in your submission bundle.

**Next:** open `04_prompt.ipynb`. It starts by loading `data/gold/<track>_<group>_gold.json`.

In [ ]:
save_json(gold, GOLD_PATH, what="gold items")

# It is git-ignored — it is your work, not part of the template. If you cloned
# into Google Drive it is already saved across sessions; if not, download it.


## Step 4 — Draw the line: dev and test

In notebook 04 you will change your prompt because of what you saw it get wrong. That is the work. But a score measured on the same items you kept adjusting against stops being a measure of how good your prompt is, and becomes a measure of **how long you kept adjusting**. It only ever goes up.

So the line gets drawn now, before anything has been run against these items:

| | what it is for |
|---|---|
| **dev** | the items you may look at. Iterate here, as many rounds as you like. |
| **test** | opened once, in the last step of notebook 04. Whatever it says is what you report. |

Both halves came out of the same sheet and the same adjudication, so the split costs you no extra annotation. What it costs is items you are allowed to learn from — which is why the ratio is a real decision and `PLAN.md` §6 asks you to defend the one you made. A bigger dev gives steadier feedback while you iterate and leaves a smaller test, so the number you finally report bounces more; a smaller dev means prompt decisions made on very few items, which is how you tune to noise and then watch the gain evaporate.

This also replaces the old advice to keep `n_per_class` at 2 while iterating. **dev is the fast set now** — a dozen or so items is about a minute per round, and your sample stays at full size throughout.

The split is stratified by label, so both halves keep every label wherever the data allows. Where it does not — a label with a single surviving item — that item goes to **test**, and the function says so. That asymmetry is deliberate: a label missing from test drops out of the macro average without announcing itself, while a label missing from dev only costs you feedback.

### The code that draws the line

SETUP imported this one; here it is, read out of `scripts/pipeline.py` when this notebook was generated. Nothing here needs running. Three things to look for: the rounding rule for a fractional `dev` is written out rather than left to `round()` (Python rounds 0.5 down and 1.5 up, and neither is something you want to have to explain in the Q&A); the rare-class clamp, and which side it favours; and the ids are **not** renumbered, because notebook 05 asks which of the model's errors are also the rows your two coders argued about, and that join runs on these ids.

**`split_dev_test`** draws the line. This is the whole of the discipline: the bookkeeping that decides whether the number in your report means anything.

```python
def split_dev_test(gold, dev, seed=42, by_document=False):
    """Split an adjudicated gold set into (dev, test).

    `dev` says how big the dev half is, and how you write it says which you meant:

        dev=3        a whole number — that many dev items per label
        dev=0.35     a decimal between 0 and 1 — that proportion of each label's items

    Both stratify by label, so every label is represented on both sides of the line
    wherever the data allows it. The same seed always gives the same split.

    Set by_document=True if you drew your sample with sample_by_document, so that no
    passage has some of its sentences in dev and the rest in test.
    """
    dev_per_class, dev_fraction = _read_dev_size(dev)

    if by_document:
        dev, test = _split_by_document(gold, dev_per_class, dev_fraction, seed)
    else:
        dev, test = _split_by_label(gold, dev_per_class, dev_fraction, seed)

    _report_split(dev, test, by_document)
    return dev, test
```

**`_read_dev_size`** is why one `dev:` setting can mean two things. A whole number is a count per label, a decimal is a proportion — the type carries the decision, so there is no second config key to keep consistent with the first.

```python
def _read_dev_size(dev):
    """Read one `dev` setting as (dev_per_class, dev_fraction) - exactly one of them set.

    A count and a proportion are one decision, so config.yaml holds one key and the way
    the number is written says which was meant: 3 is three items per label, 0.35 is a
    third of each label. The two names survive below this line only because the split
    functions need to tell the cases apart.
    """
    # bool before int: True is an int in Python, and `dev: yes` in YAML would otherwise
    # be read as "1 dev item per label" without complaining.
    if isinstance(dev, bool) or not isinstance(dev, (int, float)):
        raise ValueError(
            "dev=" + repr(dev) + " is neither a count nor a proportion.\n"
            + _DEV_HELP)
    if isinstance(dev, float):
        if not 0 < dev < 1:
            raise ValueError(
                "dev=" + str(dev) + ". A decimal is read as a proportion of each label, "
                "so it has to be strictly between 0 and 1. For a fixed count per label, "
                "write it without a decimal point.\n" + _DEV_HELP)
        return None, dev
    if dev < 1:
        raise ValueError(
            "dev=" + str(dev) + " leaves no dev set at all.\n" + _DEV_HELP)
    return dev, None
```


Now we draw the line: which of your gold items you are allowed to look at while iterating, and which you are not. Nothing here existed in Days 1–3 — no set there was worth holding back. `split_dev_test` is in `scripts/pipeline.py`.

**Run this once, and before you open notebook 04.** Splitting again after you have iterated on dev means the held-out items have already been seen — by you, if not by the model.

How big dev is comes from `dev:` in `config.yaml`, and how you wrote the number says what you meant. A balanced draw (`sample_pool`) suits a whole number, `dev: 3` — three items per label. An uneven one (`sample_random`) suits a decimal, `dev: 0.35` — a third of each label, because a fixed 3 per class would eat a small class whole.

Nothing is saved yet — read the counts it prints first.

In [ ]:
# ══ STEP 4 · Split dev / test ═════════════════════════════════════════════
# Splits your gold set in two, keeping every label on both sides wherever the
# data allows, and prints how many items each half got.
# Creates: dev, test

# ✏️ this runs as written — the work is deciding whether it should

# DEV comes from config.yaml: a whole number is items per label, a decimal
# is a proportion of each label.
#
# Drew your sample with sample_by_document? Add by_document=True inside the
# brackets below, so that no passage has some of its sentences in dev and
# the rest in test.
dev, test = split_dev_test(gold, DEV, seed=SEED)


### Now save both halves

Read the per-label counts the split just printed **before** you run this. A label that lands in dev but not in test cannot appear in the score you report, and this is the last easy moment to change `dev` in `config.yaml` and draw the line again.

Once you are happy, save. Notebook 04 opens `dev`; notebook 05 opens `test`.

In [ ]:
save_json(dev,  DEV_PATH,  what="dev items")
save_json(test, TEST_PATH, what="test items")

---

## 🛑 The `PLAN.md` gate

Notebook 04 starts calling the model. **Do not open it until your `PLAN.md` has been read and signed off.** It takes two minutes and it is not busywork: a mismatched label set or an unstated sampling seed costs an hour to unpick *after* you have burned quota on it.

Check, out loud, that these three agree: the label set in `PLAN.md`, the labels `label_set` actually returned above, and the labels your prompt file names. And that `PLAN.md` records **which sampling strategy you chose, and why**, and **§6: which split spec you set, the sizes it produced, and why that ratio**.